In [36]:
import operator
from typing import TypedDict, List, Annotated

from pydantic import BaseModel, Field
from langgraph.graph import StateGraph, START, END
from langgraph.types import Send
import os
%pip install -q langchain-openai

from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage

import psycopg
from psycopg.rows import dict_row
from langgraph.checkpoint.postgres import PostgresSaver
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv())

Note: you may need to restart the kernel to use updated packages.


True

In [37]:
def get_database_url():
    database_url = os.getenv("DATABASE_URL")

    if not database_url:
        raise ValueError(
            "DATABASE_URL is missing. Please add your Render PostgreSQL External Database URL to .env"
        )

    if "sslmode=" not in database_url:
        separator = "&" if "?" in database_url else "?"
        database_url = f"{database_url}{separator}sslmode=require"

    return database_url

In [38]:
class Task(BaseModel):
    id: int = Field(..., description="Sequential section number (1, 2, 3...)")
    title: str = Field(..., description="Authoritative, descriptive H2 title")
    brief: str = Field(..., description="Comprehensive blueprint of specific mechanisms, concepts, and nuances to cover")
    target_word_count: int = Field(
        default=450, 
        description="Target word count for depth (typically 400-700 words)"
    )
    key_takeaways: List[str] = Field(
        ..., 
        description="2-4 technical concepts, terms, or practical insights that must be explained"
    )
    section_role: str = Field(
        ..., 
        description="Structural function: 'foundational_concept', 'deep_technical_dive', 'architectural_walkthrough', or 'production_considerations'"
    )
    include_code_or_math: bool = Field(
        default=False, 
        description="True if this section requires concrete Python/PyTorch code blocks, ASCII tensor diagrams, or mathematical formulations"
    )

In [39]:
class Plan(BaseModel):
    title: str = Field(..., description="High-impact, SEO-optimized title without clickbait fluff")
    target_audience: str = Field(..., description="Target reader persona (e.g., Applied ML Engineers, Senior Software Engineers)")
    technical_depth: str = Field(..., description="Level: Intermediate, Advanced, or Production-Grade")
    tasks: List[Task] = Field(..., description="5-7 logically progressing sections")

In [40]:
class State(TypedDict):
    topic:str
    plan:Plan
    sections:Annotated[List[str],operator.add ]
    final:str



In [41]:


openrouter_api_key = os.getenv("OPENROUTER_API_KEY")

if not openrouter_api_key:
    raise ValueError("GROQ_API_KEY is missing. Please add it to your .env file.")

openrouter_api_key = os.getenv("OPENROUTER_API_KEY")

if not openrouter_api_key:
    raise ValueError(
        "OPENROUTER_API_KEY is missing. Please add it to your .env file."
    )

model = ChatOpenAI(
    model="meta-llama/llama-3.3-70b-instruct:free",
    api_key=openrouter_api_key,
    base_url="https://openrouter.ai/api/v1",
    temperature=0.7,
)
openrouter_api_key = os.getenv("OPENROUTER_API_KEY")

if not openrouter_api_key:
        raise ValueError("OPENROUTER_API_KEY is missing. Please add it to your .env file.")

model = ChatOpenAI(
        model="openrouter/free",
        api_key=openrouter_api_key,
        base_url="https://openrouter.ai/api/v1",
        temperature=0.7,
    )
     

In [42]:
ORCHESTRATOR_SYSTEM_PROMPT = """You are a Principal AI Research Engineer and Executive Technical Editor for premier engineering publications (like Distill, Google DeepMind Research, or PyTorch blogs).

Your task is to decompose a given technical topic into a comprehensive 5 to 7 section blog blueprint.

Strict Architecture & Planning Rules:
1. Narrative Arc: Structure the post logically:
   - Root Problem / Historical Bottleneck
   - Core Theoretical Foundations & Mathematical Formulations
   - Structural Anatomy & Mechanism Walkthrough
   - Concrete Implementation Patterns (real PyTorch/Python mechanics)
   - Edge Cases, Failure Modes, Scaling Bottlenecks & Production Optimizations
2. Exhaustive Briefs: Each task brief must be packed with specific domain terminology, trade-offs, and exact equations or mechanisms to analyze.
3. No Fluff: Explicitly ban introductory hand-waving and generic statements like "In today's fast-paced world."
4. Concrete Roles: Explicitly flag sections that require mathematical formulas (LaTeX) or executable code snippets.
"""

def orchestrator(state: State) -> dict:
    structured_planner = model.with_structured_output(Plan)
    plan = structured_planner.invoke(
        [
            SystemMessage(content=ORCHESTRATOR_SYSTEM_PROMPT),
            HumanMessage(
                content=(
                    f"Topic: {state['topic']}\n\n"
                    "Generate a rigorous, publication-grade blueprint."
                )
            ),
        ]
    )
    return {"plan": plan}

In [43]:
def fanout(state: State):
    return [
        Send("worker", {"task": task, "topic": state["topic"], "plan": state["plan"]})
        for task in state["plan"].tasks
    ]

In [44]:
WORKER_SYSTEM_PROMPT = """You are a Staff Software Engineer and Technical Author writing a single authoritative section for a master engineering guide.

Formatting & Style Rules:
- Direct Opening: Dive straight into technical substance in sentence 1. Never begin with conversational filler ("In this section...", "Let's explore...", "Now that we covered...").
- Exact Specifics Over Abstractions: Use precise variables, tensor dimensions (e.g., `(batch_size, seq_len, d_model)`), and algorithmic complexities ($O(N^2)$ vs $O(N)$).
- Code & Math: If requested, provide runnable Python/PyTorch code or formal LaTeX formulas enclosed in $...$ or $$...$$. Never pseudo-code.
- Structure: Start with `## {section_title}`. Use `###` for sub-arguments, bulleted lists for trade-offs, and bold markers for technical invariants.
- No Section Conclusions: Do NOT add a summary or conclusion paragraph at the end of the section; end on a concrete technical insight.
"""

def worker(payload: dict) -> dict:
    task: Task = payload["task"]
    plan: Plan = payload["plan"]
    topic: str = payload["topic"]

    takeaways_formatted = "\n".join(f"- {item}" for item in task.key_takeaways)

    prompt = (
        f"Document Title: {plan.title}\n"
        f"Target Audience: {plan.target_audience} (Depth: {plan.technical_depth})\n"
        f"Overarching Topic: {topic}\n"
        f"-----------------------------------------\n"
        f"Current Section ID: {task.id}\n"
        f"Current Section Title: {task.title}\n"
        f"Section Role: {task.section_role}\n"
        f"Target Length: ~{task.target_word_count} words\n"
        f"Include Code/Math: {task.include_code_or_math}\n\n"
        f"Detailed Blueprint to Execute:\n{task.brief}\n\n"
        f"Mandatory Inclusions:\n{takeaways_formatted}\n\n"
        "Write only the Markdown content for this section now."
    )

    response = model.invoke(
        [
            SystemMessage(content=WORKER_SYSTEM_PROMPT),
            HumanMessage(content=prompt),
        ]
    )

    return {"sections": [response.content.strip()]}

In [45]:
import re
from pathlib import Path

def reducer(state: State) -> dict:
    plan: Plan = state["plan"]
    title = getattr(plan, "title", "Technical Blog Post")

    meta_header = (
        f"# {title}\n\n"
        f"> **Audience:** {getattr(plan, 'target_audience', 'Engineers')} | "
        f"**Level:** {getattr(plan, 'technical_depth', 'Advanced')}\n\n"
    )

    body = "\n\n---\n\n".join(state.get("sections", [])).strip()
    final_md = f"{meta_header}\n{body}\n"

    clean_title = re.sub(r'[\\/*?:"<>|]', "", title).strip().lower().replace(" ", "_")
    filename = f"{clean_title}.md"

    output_path = Path(filename)
    output_path.write_text(final_md, encoding="utf-8")

    return {"final": final_md}

In [46]:
graph = StateGraph(State)
graph.add_node("orchestrator", orchestrator)
graph.add_node("worker", worker)
graph.add_node("reducer", reducer)

graph.add_edge(START, "orchestrator")
graph.add_conditional_edges("orchestrator", fanout, ["worker"])
graph.add_edge("worker", "reducer")
graph.add_edge("reducer", END)

In [47]:
DATABASE_URL=get_database_url
DATABASE_URL = get_database_url()

_conn = psycopg.connect(
    DATABASE_URL,
    autocommit=True,
    row_factory=dict_row,
)

checkpointer = PostgresSaver(_conn)
checkpointer.setup()
checkpointer=PostgresSaver(_conn)
checkpointer.setup()

In [48]:
workflow=graph.compile(checkpointer=checkpointer)

In [49]:
import uuid

config = {"configurable": {"thread_id": str(uuid.uuid4())}}

out = workflow.invoke(
    {
        "topic": "write a deep dive technical blog on transformers, self-attention mechanisms, and scaling limits",
        "sections": []
    },
    config=config
)

print(f"Done! Character length: {len(out['final'])}")

Done! Character length: 19980
